In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
from prophet import Prophet

# Deep Learning Libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import optuna

# Plotting
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("=" * 80)
print("METHANE TIME SERIES FORECASTING SYSTEM")
print("TRULY ROBUST VERSION - WORKS ACROSS ALL DATASETS")
print("=" * 80)

# ============================================================================
# PART 1: DATA LOADING AND PREPROCESSING
# ============================================================================

import os
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
file_path = os.path.join(desktop_path, "All.xlsx")

print(f"\n[1/10] Loading data from: {file_path}")
df = pd.read_excel(file_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData statistics:")
print(df.describe())

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Fill missing values with interpolation
for col in df.columns:
    if col != 'Date' and df[col].isnull().any():
        df[col] = df[col].interpolate(method='linear', limit_direction='both')

# Remove any remaining NaN and outliers
df = df.dropna().reset_index(drop=True)

# Remove extreme outliers using IQR method (keeps data robust)
for col in ['windspeed', 'RelHumidity', 'dewpointtemp', 'Airtemp', 'methane']:
    Q1 = df[col].quantile(0.01)
    Q3 = df[col].quantile(0.99)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

df = df.reset_index(drop=True)
print(f"Data after cleaning and outlier removal: {df.shape}")

# ============================================================================
# PART 2: ADAPTIVE LAG CORRELATION ANALYSIS
# ============================================================================

print("\n[2/10] Analyzing lag correlation between Methane and Air Temperature...")

# Adaptive max lag based on dataset size (more data = can test longer lags)
max_lag = min(60, len(df) // 20)  # Adaptive: test up to 60 days or 5% of data
correlations = []

for lag in range(max_lag + 1):
    df_temp = df.copy()
    df_temp[f'Airtemp_lag_{lag}'] = df_temp['Airtemp'].shift(lag)
    valid_data = df_temp[['methane', f'Airtemp_lag_{lag}']].dropna()
    if len(valid_data) > 0:
        corr = valid_data['methane'].corr(valid_data[f'Airtemp_lag_{lag}'])
        correlations.append({'lag': lag, 'correlation': corr})

corr_df = pd.DataFrame(correlations)
corr_df = corr_df.dropna()
best_lag = int(corr_df.loc[corr_df['correlation'].abs().idxmax(), 'lag'])
best_corr = corr_df.loc[corr_df['correlation'].abs().idxmax(), 'correlation']

print(f"Best lag: {best_lag} days")
print(f"Best correlation: {best_corr:.4f}")

# Apply best lag to all features
print("\nApplying optimal lag to all features...")
for col in ['windspeed', 'RelHumidity', 'dewpointtemp', 'Airtemp']:
    df[f'{col}_lag_{best_lag}'] = df[col].shift(best_lag)

df = df.dropna().reset_index(drop=True)
print(f"Data shape after lagging: {df.shape}")

# ============================================================================
# PART 3: ADAPTIVE FEATURE ENGINEERING
# ============================================================================

print("\n[3/10] Engineering adaptive features...")

# Time-based features
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df['dayofweek'] = df['Date'].dt.dayofweek
df['dayofyear'] = df['Date'].dt.dayofyear
df['quarter'] = df['Date'].dt.quarter
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# Cyclical encoding (preserves seasonality)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

# Adaptive rolling windows based on dataset size
dataset_days = (df['Date'].max() - df['Date'].min()).days
if dataset_days > 365:  # More than 1 year of data
    windows = [3, 7, 14, 30, 60]
elif dataset_days > 180:  # 6 months to 1 year
    windows = [3, 7, 14, 30]
else:  # Less than 6 months
    windows = [3, 7, 14]

print(f"Using rolling windows: {windows} (adapted to {dataset_days} days of data)")

for window in windows:
    df[f'methane_roll_mean_{window}'] = df['methane'].rolling(window=window, min_periods=1).mean()
    df[f'methane_roll_std_{window}'] = df['methane'].rolling(window=window, min_periods=1).std().fillna(0)
    df[f'Airtemp_roll_mean_{window}'] = df['Airtemp'].rolling(window=window, min_periods=1).mean()
    df[f'Airtemp_roll_std_{window}'] = df['Airtemp'].rolling(window=window, min_periods=1).std().fillna(0)

# Interaction features (physical relationships)
df['temp_diff'] = df['Airtemp'] - df['dewpointtemp']
df['temp_humidity_interaction'] = df['Airtemp'] * df['RelHumidity'] / 100
df['temp_wind_interaction'] = df['Airtemp'] * df['windspeed']

# Exponential moving averages (smoother trends)
ema_spans = [7, 30] if dataset_days > 60 else [7]
for span in ema_spans:
    df[f'methane_ema_{span}'] = df['methane'].ewm(span=span, adjust=False).mean()
    df[f'Airtemp_ema_{span}'] = df['Airtemp'].ewm(span=span, adjust=False).mean()

df = df.dropna().reset_index(drop=True)
print(f"Total features created: {len(df.columns) - 2}")
print(f"Final data shape: {df.shape}")

# ============================================================================
# PART 4: ADAPTIVE TRAIN/VALIDATION/TEST SPLIT
# ============================================================================

print("\n[4/10] Preparing adaptive train/validation split...")

train_df = df[df['Date'] < '2025-01-01'].copy()
print(f"Training data: {train_df['Date'].min()} to {train_df['Date'].max()}")
print(f"Training samples: {len(train_df)}")

# Define feature columns
feature_cols = [col for col in df.columns if col not in ['Date', 'methane']]
target_col = 'methane'

X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

# Remove any NaN
valid_idx = ~(np.isnan(X_train).any(axis=1) | np.isnan(y_train))
X_train = X_train[valid_idx]
y_train = y_train[valid_idx]
train_df = train_df[valid_idx].reset_index(drop=True)

print(f"Clean training samples: {len(X_train)}")

# Use RobustScaler (resistant to outliers)
scaler_X = RobustScaler()
scaler_y = RobustScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()

# Adaptive lookback based on data frequency
lookback = min(30, len(X_train) // 10)  # 30 days or 10% of data
print(f"Using lookback window: {lookback} days")

# ============================================================================
# PART 5: MODEL 1 - PROPHET (OPTIMIZED)
# ============================================================================

print("\n[5/10] Optimizing Prophet model...")

def objective_prophet(trial):
    """Optimize Prophet hyperparameters"""
    prophet_df = train_df[['Date', 'methane']].copy()
    prophet_df.columns = ['ds', 'y']
    
    for col in ['Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']:
        prophet_df[col] = train_df[col].values
    
    # Split for validation
    train_size = int(len(prophet_df) * 0.8)
    train_prophet = prophet_df[:train_size]
    val_prophet = prophet_df[train_size:]
    
    params = {
        'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True),
        'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True),
        'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
        'changepoint_range': trial.suggest_float('changepoint_range', 0.8, 0.95),
    }
    
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        **params
    )
    
    for col in ['Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']:
        model.add_regressor(col, prior_scale=1.0, mode='additive')
    
    model.fit(train_prophet)
    
    # Validate
    forecast = model.predict(val_prophet)
    y_true = val_prophet['y'].values
    y_pred = forecast['yhat'].values
    
    return r2_score(y_true, y_pred)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_prophet = optuna.create_study(direction='maximize', study_name='prophet_optimize')
study_prophet.optimize(objective_prophet, n_trials=30, show_progress_bar=False)

print(f"Best Prophet score: {study_prophet.best_value:.4f}")
print(f"Best parameters: {study_prophet.best_params}")

# Train final Prophet model
prophet_df = train_df[['Date', 'methane']].copy()
prophet_df.columns = ['ds', 'y']
for col in ['Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']:
    prophet_df[col] = train_df[col].values

prophet_model = Prophet(**study_prophet.best_params, yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
for col in ['Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']:
    prophet_model.add_regressor(col, prior_scale=1.0, mode='additive')
prophet_model.fit(prophet_df)

print("Prophet model trained successfully!")

# ============================================================================
# PART 6: MODEL 2 - XGBOOST (OPTIMIZED)
# ============================================================================

print("\n[6/10] Optimizing XGBoost...")

def objective_xgb(trial):
    """Optimize XGBoost hyperparameters"""
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 3.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 3.0),
    }
    
    # Adaptive number of splits based on data size
    n_splits = min(5, len(X_train) // 100)
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    
    for train_idx, val_idx in tscv.split(X_train_scaled):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_tr, y_val = y_train_scaled[train_idx], y_train_scaled[val_idx]
        
        model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, early_stopping_rounds=50)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        
        pred = model.predict(X_val)
        score = r2_score(y_val, pred)
        scores.append(score)
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    return mean_score - 0.1 * std_score

study_xgb = optuna.create_study(direction='maximize', study_name='xgboost_optimize')
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=False)

print(f"Best XGBoost score: {study_xgb.best_value:.4f}")
print(f"Best parameters: {study_xgb.best_params}")

xgb_model = xgb.XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train_scaled)
print("XGBoost model trained successfully!")

# ============================================================================
# PART 7: MODEL 3 - N-BEATS (OPTIMIZED)
# ============================================================================

print("\n[7/10] Optimizing N-BEATS...")

class NBEATSBlock(nn.Module):
    def __init__(self, input_size, theta_size, horizon, n_neurons, n_layers, dropout=0.2):
        super().__init__()
        self.input_size = input_size
        self.theta_size = theta_size
        self.horizon = horizon
        
        layers = [nn.Linear(input_size, n_neurons), nn.ReLU(), nn.Dropout(dropout)]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(n_neurons, n_neurons), nn.ReLU(), nn.Dropout(dropout)])
        
        self.fc_stack = nn.Sequential(*layers)
        self.theta_f = nn.Linear(n_neurons, theta_size)
        self.theta_b = nn.Linear(n_neurons, theta_size)
        
    def forward(self, x):
        x = self.fc_stack(x)
        theta_f = self.theta_f(x)
        theta_b = self.theta_b(x)
        return theta_f, theta_b

class NBEATS(nn.Module):
    def __init__(self, input_size, horizon, n_blocks=2, n_layers=3, n_neurons=128, dropout=0.2):
        super().__init__()
        self.input_size = input_size
        self.horizon = horizon
        
        self.blocks = nn.ModuleList([
            NBEATSBlock(input_size, horizon, horizon, n_neurons, n_layers, dropout)
            for _ in range(n_blocks)
        ])
        
    def forward(self, x):
        forecast = torch.zeros(x.size(0), self.horizon).to(x.device)
        for block in self.blocks:
            theta_f, theta_b = block(x)
            forecast = forecast + theta_f
        return forecast

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback=30):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.lookback = lookback
        
    def __len__(self):
        return len(self.X) - self.lookback
    
    def __getitem__(self, idx):
        return self.X[idx:idx+self.lookback].flatten(), self.y[idx+self.lookback]

def objective_nbeats(trial):
    """Optimize N-BEATS hyperparameters"""
    n_blocks = trial.suggest_int('n_blocks', 1, 3)
    n_layers = trial.suggest_int('n_layers', 2, 4)
    n_neurons = trial.suggest_categorical('n_neurons', [64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    
    dataset = TimeSeriesDataset(X_train_scaled, y_train_scaled, lookback=lookback)
    train_size = int(len(dataset) * 0.8)
    train_dataset = torch.utils.data.Subset(dataset, range(train_size))
    val_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = NBEATS(
        input_size=lookback * X_train_scaled.shape[1],
        horizon=1,
        n_blocks=n_blocks,
        n_layers=n_layers,
        n_neurons=n_neurons,
        dropout=dropout
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    
    # Train for limited epochs
    for epoch in range(20):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch).squeeze()
            loss = criterion(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    
    # Validate
    model.eval()
    val_preds = []
    val_true = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            output = model(X_batch).squeeze()
            val_preds.extend(output.cpu().numpy())
            val_true.extend(y_batch.numpy())
    
    return r2_score(val_true, val_preds)

study_nbeats = optuna.create_study(direction='maximize', study_name='nbeats_optimize')
study_nbeats.optimize(objective_nbeats, n_trials=20, show_progress_bar=False)

print(f"Best N-BEATS score: {study_nbeats.best_value:.4f}")
print(f"Best parameters: {study_nbeats.best_params}")

# Train final N-BEATS model
dataset = TimeSeriesDataset(X_train_scaled, y_train_scaled, lookback=lookback)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

nbeats_model = NBEATS(
    input_size=lookback * X_train_scaled.shape[1],
    horizon=1,
    n_blocks=study_nbeats.best_params['n_blocks'],
    n_layers=study_nbeats.best_params['n_layers'],
    n_neurons=study_nbeats.best_params['n_neurons'],
    dropout=study_nbeats.best_params['dropout']
).to(device)

optimizer = torch.optim.Adam(nbeats_model.parameters(), lr=study_nbeats.best_params['lr'], weight_decay=1e-5)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

n_epochs = 100
best_loss = float('inf')
patience_counter = 0
patience = 15

for epoch in range(n_epochs):
    nbeats_model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = nbeats_model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nbeats_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss)
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}")

print("N-BEATS model trained successfully!")

# ============================================================================
# PART 8: MODEL 4 - DLINEAR (OPTIMIZED)
# ============================================================================

print("\n[8/10] Optimizing DLinear...")

class DLinear(nn.Module):
    def __init__(self, input_size, horizon, dropout=0.2):
        super().__init__()
        self.input_size = input_size
        self.horizon = horizon
        
        self.linear_seasonal = nn.Linear(input_size, horizon)
        self.linear_trend = nn.Linear(input_size, horizon)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        seasonal_init = x - x.mean(dim=1, keepdim=True)
        trend_init = x.mean(dim=1, keepdim=True).expand_as(x)
        
        seasonal_init = self.dropout(seasonal_init)
        trend_init = self.dropout(trend_init)
        
        seasonal_output = self.linear_seasonal(seasonal_init)
        trend_output = self.linear_trend(trend_init)
        
        return seasonal_output + trend_output

def objective_dlinear(trial):
    """Optimize DLinear hyperparameters"""
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    
    dataset = TimeSeriesDataset(X_train_scaled, y_train_scaled, lookback=lookback)
    train_size = int(len(dataset) * 0.8)
    train_dataset = torch.utils.data.Subset(dataset, range(train_size))
    val_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DLinear(
        input_size=lookback * X_train_scaled.shape[1],
        horizon=1,
        dropout=dropout
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    
    for epoch in range(20):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch).squeeze()
            loss = criterion(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    
    model.eval()
    val_preds = []
    val_true = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            output = model(X_batch).squeeze()
            val_preds.extend(output.cpu().numpy())
            val_true.extend(y_batch.numpy())
    
    return r2_score(val_true, val_preds)

study_dlinear = optuna.create_study(direction='maximize', study_name='dlinear_optimize')
study_dlinear.optimize(objective_dlinear, n_trials=20, show_progress_bar=False)

print(f"Best DLinear score: {study_dlinear.best_value:.4f}")
print(f"Best parameters: {study_dlinear.best_params}")

# Train final DLinear model
dlinear_model = DLinear(
    input_size=lookback * X_train_scaled.shape[1],
    horizon=1,
    dropout=study_dlinear.best_params['dropout']
).to(device)

optimizer = torch.optim.Adam(dlinear_model.parameters(), lr=study_dlinear.best_params['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

best_loss = float('inf')
patience_counter = 0

for epoch in range(n_epochs):
    dlinear_model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        output = dlinear_model(X_batch).squeeze()
        loss = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dlinear_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    scheduler.step(avg_loss)
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {avg_loss:.6f}")

print("DLinear model trained successfully!")
# ============================================================================
# SCATTER PLOT: ACTUAL VS PREDICTED (BEST MODEL ON TRAINING DATA)
# ============================================================================

print("\n[BONUS] Creating scatter plot for model validation...")

# Use last 20% of training data for visualization
split_point = int(len(train_df) * 0.8)
val_visual_df = train_df[split_point:].copy().reset_index(drop=True)

print(f"Creating scatter plot with {len(val_visual_df)} validation samples...")

X_val_vis = val_visual_df[feature_cols].values
y_val_vis = val_visual_df[target_col].values
X_val_vis_scaled = scaler_X.transform(X_val_vis)

# Get predictions from all models
print("Getting Prophet predictions...")
prophet_val_vis = val_visual_df[['Date', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']].copy()
prophet_val_vis.columns = ['ds', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']
prophet_pred_vis = prophet_model.predict(prophet_val_vis)['yhat'].values

print("Getting XGBoost predictions...")
xgb_pred_vis_scaled = xgb_model.predict(X_val_vis_scaled)
xgb_pred_vis = scaler_y.inverse_transform(xgb_pred_vis_scaled.reshape(-1, 1)).flatten()

print("Getting N-BEATS predictions...")
nbeats_model.eval()
nbeats_preds_vis = []
with torch.no_grad():
    for i in range(len(X_val_vis_scaled)):
        actual_idx = split_point + i
        if actual_idx < lookback:
            X_input = X_train_scaled[max(0, actual_idx-lookback):actual_idx]
            if len(X_input) < lookback:
                padding = np.repeat(X_train_scaled[0:1], lookback - len(X_input), axis=0)
                X_input = np.concatenate([padding, X_input])
        else:
            X_input = X_train_scaled[actual_idx-lookback:actual_idx]
        
        X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
        pred = nbeats_model(X_tensor).cpu().numpy()[0, 0]
        nbeats_preds_vis.append(pred)

nbeats_pred_vis = scaler_y.inverse_transform(np.array(nbeats_preds_vis).reshape(-1, 1)).flatten()

print("Getting DLinear predictions...")
dlinear_model.eval()
dlinear_preds_vis = []
with torch.no_grad():
    for i in range(len(X_val_vis_scaled)):
        actual_idx = split_point + i
        if actual_idx < lookback:
            X_input = X_train_scaled[max(0, actual_idx-lookback):actual_idx]
            if len(X_input) < lookback:
                padding = np.repeat(X_train_scaled[0:1], lookback - len(X_input), axis=0)
                X_input = np.concatenate([padding, X_input])
        else:
            X_input = X_train_scaled[actual_idx-lookback:actual_idx]
        
        X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
        pred = dlinear_model(X_tensor).cpu().numpy()[0, 0]
        dlinear_preds_vis.append(pred)

dlinear_pred_vis = scaler_y.inverse_transform(np.array(dlinear_preds_vis).reshape(-1, 1)).flatten()

# Calculate R2 for each model
print("\nCalculating R² scores...")
r2_prophet = r2_score(y_val_vis, prophet_pred_vis)
r2_xgb = r2_score(y_val_vis, xgb_pred_vis)
r2_nbeats = r2_score(y_val_vis, nbeats_pred_vis)
r2_dlinear = r2_score(y_val_vis, dlinear_pred_vis)

print(f"Prophet R²: {r2_prophet:.4f}")
print(f"XGBoost R²: {r2_xgb:.4f}")
print(f"N-BEATS R²: {r2_nbeats:.4f}")
print(f"DLinear R²: {r2_dlinear:.4f}")

# Determine best model
r2_scores = {
    'Prophet': r2_prophet, 
    'XGBoost': r2_xgb, 
    'N-BEATS': r2_nbeats, 
    'DLinear': r2_dlinear
}
best_model_name = max(r2_scores, key=r2_scores.get)
best_r2 = r2_scores[best_model_name]

# Get predictions from best model
prediction_map = {
    'Prophet': prophet_pred_vis,
    'XGBoost': xgb_pred_vis,
    'N-BEATS': nbeats_pred_vis,
    'DLinear': dlinear_pred_vis
}
best_pred = prediction_map[best_model_name]

print(f"\nBest model: {best_model_name} with R²={best_r2:.4f}")
print(f"Creating scatter plot...")

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_val_vis, best_pred, alpha=0.6, s=50, c='blue', edgecolors='black', linewidth=0.5)

# Add perfect prediction line
min_val = min(y_val_vis.min(), best_pred.min())
max_val = max(y_val_vis.max(), best_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

# Add trend line
z = np.polyfit(y_val_vis, best_pred, 1)
p = np.poly1d(z)
sorted_y = np.sort(y_val_vis)
ax.plot(sorted_y, p(sorted_y), 'g-', linewidth=2, alpha=0.7, label=f'Trend: y={z[0]:.3f}x+{z[1]:.3f}')

ax.set_xlabel('Actual Methane (ppm)', fontsize=14, fontweight='bold')
ax.set_ylabel('Predicted Methane (ppm)', fontsize=14, fontweight='bold')
ax.set_title(f'Actual vs Predicted Methane - {best_model_name}\nR² = {best_r2:.4f}', 
          fontsize=16, fontweight='bold')
ax.legend(fontsize=12, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Save plot
plot_path = os.path.join(desktop_path, f"methane_scatter_plot_{best_model_name}.jpg")
plt.savefig(plot_path, dpi=300, format='jpg', bbox_inches='tight')
print(f"\n✓ SCATTER PLOT SAVED: {plot_path}")
plt.close()

print(f"  File location: {plot_path}")
print(f"  Best model: {best_model_name}")
print(f"  R² score: {best_r2:.4f}")
print(f"  Number of points: {len(y_val_vis)}")

# ============================================================================
# PART 9: VALIDATION WITH MULTIPLE METRICS
# ============================================================================

print("\n[9/10] Calculating validation metrics...")

def calculate_metrics(y_true, y_pred):
    """Calculate metrics with NaN handling"""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred) | np.isinf(y_true) | np.isinf(y_pred))
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]
    
    if len(y_true_clean) == 0:
        return {'R2': 0.0, 'RMSE': 999.0, 'MAE': 999.0, 'MAPE': 999.0}
    
    r2 = r2_score(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    
    mask_nonzero = y_true_clean != 0
    if mask_nonzero.sum() > 0:
        mape = np.mean(np.abs((y_true_clean[mask_nonzero] - y_pred_clean[mask_nonzero]) / y_true_clean[mask_nonzero])) * 100
    else:
        mape = 999.0
    
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

# Multiple validation periods for robustness
validation_periods = [
    ('Last 6 months', train_df['Date'].max() - pd.Timedelta(days=180)),
    ('Last 3 months', train_df['Date'].max() - pd.Timedelta(days=90)),
    ('Last 1 month', train_df['Date'].max() - pd.Timedelta(days=30))
]

all_validation_results = []

for period_name, validation_start in validation_periods:
    val_df = train_df[train_df['Date'] >= validation_start].copy()
    
    if len(val_df) <= lookback:
        continue
    
    print(f"\n{period_name}: {val_df['Date'].min()} to {val_df['Date'].max()} ({len(val_df)} samples)")
    
    X_val = val_df[feature_cols].values
    y_val = val_df[target_col].values
    X_val_scaled = scaler_X.transform(X_val)
    
    # Prophet
    prophet_val = val_df[['Date', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']].copy()
    prophet_val.columns = ['ds', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']
    prophet_val_pred = prophet_model.predict(prophet_val)['yhat'].values
    prophet_metrics = calculate_metrics(y_val, prophet_val_pred)
    
    # XGBoost
    xgb_val_pred_scaled = xgb_model.predict(X_val_scaled)
    xgb_val_pred = scaler_y.inverse_transform(xgb_val_pred_scaled.reshape(-1, 1)).flatten()
    xgb_metrics = calculate_metrics(y_val, xgb_val_pred)
    
    # N-BEATS
    nbeats_model.eval()
    nbeats_val_preds = []
    with torch.no_grad():
        val_start_idx = len(X_train_scaled) - len(X_val_scaled)
        for i in range(len(X_val_scaled)):
            actual_idx = val_start_idx + i
            if actual_idx < lookback:
                X_input = X_train_scaled[max(0, actual_idx-lookback):actual_idx+1]
                if len(X_input) < lookback:
                    padding = np.repeat(X_input[0:1], lookback - len(X_input), axis=0)
                    X_input = np.concatenate([padding, X_input])
            else:
                X_input = X_train_scaled[actual_idx-lookback:actual_idx]
            
            X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
            pred = nbeats_model(X_tensor).cpu().numpy()[0, 0]
            nbeats_val_preds.append(pred)
    
    nbeats_val_pred = scaler_y.inverse_transform(np.array(nbeats_val_preds).reshape(-1, 1)).flatten()
    nbeats_metrics = calculate_metrics(y_val, nbeats_val_pred)
    
    # DLinear
    dlinear_model.eval()
    dlinear_val_preds = []
    with torch.no_grad():
        val_start_idx = len(X_train_scaled) - len(X_val_scaled)
        for i in range(len(X_val_scaled)):
            actual_idx = val_start_idx + i
            if actual_idx < lookback:
                X_input = X_train_scaled[max(0, actual_idx-lookback):actual_idx+1]
                if len(X_input) < lookback:
                    padding = np.repeat(X_input[0:1], lookback - len(X_input), axis=0)
                    X_input = np.concatenate([padding, X_input])
            else:
                X_input = X_train_scaled[actual_idx-lookback:actual_idx]
            
            X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
            pred = dlinear_model(X_tensor).cpu().numpy()[0, 0]
            dlinear_val_preds.append(pred)
    
    dlinear_val_pred = scaler_y.inverse_transform(np.array(dlinear_val_preds).reshape(-1, 1)).flatten()
    dlinear_metrics = calculate_metrics(y_val, dlinear_val_pred)
    
    # Ensemble (weighted by inverse RMSE)
    weights = np.array([
        1/prophet_metrics['RMSE'] if prophet_metrics['RMSE'] > 0 else 0,
        1/xgb_metrics['RMSE'] if xgb_metrics['RMSE'] > 0 else 0,
        1/nbeats_metrics['RMSE'] if nbeats_metrics['RMSE'] > 0 else 0,
        1/dlinear_metrics['RMSE'] if dlinear_metrics['RMSE'] > 0 else 0
    ])
    weights = weights / weights.sum()
    
    ensemble_val_pred = (
        weights[0] * prophet_val_pred + 
        weights[1] * xgb_val_pred + 
        weights[2] * nbeats_val_pred + 
        weights[3] * dlinear_val_pred
    )
    ensemble_metrics = calculate_metrics(y_val, ensemble_val_pred)
    
    period_results = pd.DataFrame({
        'Period': [period_name] * 5,
        'Model': ['Prophet', 'XGBoost', 'N-BEATS', 'DLinear', 'Ensemble'],
        'R2': [prophet_metrics['R2'], xgb_metrics['R2'], nbeats_metrics['R2'], 
               dlinear_metrics['R2'], ensemble_metrics['R2']],
        'RMSE': [prophet_metrics['RMSE'], xgb_metrics['RMSE'], nbeats_metrics['RMSE'],
                dlinear_metrics['RMSE'], ensemble_metrics['RMSE']],
        'MAE': [prophet_metrics['MAE'], xgb_metrics['MAE'], nbeats_metrics['MAE'],
               dlinear_metrics['MAE'], ensemble_metrics['MAE']],
        'MAPE': [prophet_metrics['MAPE'], xgb_metrics['MAPE'], nbeats_metrics['MAPE'],
                dlinear_metrics['MAPE'], ensemble_metrics['MAPE']]
    })
    
    print(period_results.to_string(index=False))
    all_validation_results.append(period_results)

validation_metrics = pd.concat(all_validation_results, ignore_index=True)

# Calculate ensemble weights from last validation period
last_val_results = validation_metrics[validation_metrics['Period'] == 'Last 1 month']
if len(last_val_results) > 0:
    rmse_values = last_val_results[last_val_results['Model'] != 'Ensemble']['RMSE'].values
    ensemble_weights = 1 / (rmse_values + 1e-10)
    ensemble_weights = ensemble_weights / ensemble_weights.sum()
    print(f"\nEnsemble weights based on validation RMSE:")
    print(f"  Prophet: {ensemble_weights[0]:.3f}")
    print(f"  XGBoost: {ensemble_weights[1]:.3f}")
    print(f"  N-BEATS: {ensemble_weights[2]:.3f}")
    print(f"  DLinear: {ensemble_weights[3]:.3f}")
else:
    ensemble_weights = np.array([0.25, 0.25, 0.25, 0.25])


# ============================================================================
# PART 10: FORECASTING WITH WEIGHTED ENSEMBLE
# ============================================================================

print("\n[10/10] Generating forecasts for 2025...")

def create_future_features(start_date, end_date, train_df, feature_cols, best_lag):
    """Create future features based on historical patterns"""
    future_df = pd.DataFrame({'Date': pd.date_range(start_date, end_date, freq='D')})
    
    month_num = start_date.month
    historical_month = train_df[train_df['Date'].dt.month == month_num]
    
    for col in ['windspeed', 'RelHumidity', 'dewpointtemp', 'Airtemp']:
        if len(historical_month) > 0:
            future_df[col] = historical_month[col].median()
        else:
            future_df[col] = train_df[col].median()
    
    # Add lag features
    for col in ['windspeed', 'RelHumidity', 'dewpointtemp', 'Airtemp']:
        future_df[f'{col}_lag_{best_lag}'] = future_df[col]
    
    # Time features
    future_df['year'] = future_df['Date'].dt.year
    future_df['month'] = future_df['Date'].dt.month
    future_df['day'] = future_df['Date'].dt.day
    future_df['dayofweek'] = future_df['Date'].dt.dayofweek
    future_df['dayofyear'] = future_df['Date'].dt.dayofyear
    future_df['quarter'] = future_df['Date'].dt.quarter
    future_df['is_weekend'] = (future_df['dayofweek'] >= 5).astype(int)
    
    # Cyclical features
    future_df['month_sin'] = np.sin(2 * np.pi * future_df['month'] / 12)
    future_df['month_cos'] = np.cos(2 * np.pi * future_df['month'] / 12)
    future_df['day_sin'] = np.sin(2 * np.pi * future_df['day'] / 31)
    future_df['day_cos'] = np.cos(2 * np.pi * future_df['day'] / 31)
    future_df['dayofyear_sin'] = np.sin(2 * np.pi * future_df['dayofyear'] / 365)
    future_df['dayofyear_cos'] = np.cos(2 * np.pi * future_df['dayofyear'] / 365)
    
    # Rolling features (use historical medians)
    for window in windows:
        future_df[f'methane_roll_mean_{window}'] = train_df['methane'].tail(window).median()
        future_df[f'methane_roll_std_{window}'] = train_df['methane'].tail(window).std()
        future_df[f'Airtemp_roll_mean_{window}'] = train_df['Airtemp'].tail(window).median()
        future_df[f'Airtemp_roll_std_{window}'] = train_df['Airtemp'].tail(window).std()
    
    # Interaction features
    future_df['temp_diff'] = future_df['Airtemp'] - future_df['dewpointtemp']
    future_df['temp_humidity_interaction'] = future_df['Airtemp'] * future_df['RelHumidity'] / 100
    future_df['temp_wind_interaction'] = future_df['Airtemp'] * future_df['windspeed']
    
    # EMA features
    for span in ema_spans:
        future_df[f'methane_ema_{span}'] = train_df['methane'].tail(span).median()
        future_df[f'Airtemp_ema_{span}'] = train_df['Airtemp'].tail(span).median()
    
    # Ensure all feature columns are present
    for col in feature_cols:
        if col not in future_df.columns:
            future_df[col] = 0
    
    return future_df

results_list = []
detailed_forecasts = []

month_dates = [
    ('2025-01-01', '2025-01-31', 1),
    ('2025-02-01', '2025-02-28', 2),
    ('2025-03-01', '2025-03-31', 3),
    ('2025-04-01', '2025-04-30', 4),
    ('2025-05-01', '2025-05-31', 5),
    ('2025-06-01', '2025-06-30', 6)
]

for start_str, end_str, month_num in month_dates:
    print(f"\nForecasting Month {month_num} (2025)...")
    
    start_date = pd.Timestamp(start_str)
    end_date = pd.Timestamp(end_str)
    
    future_df = create_future_features(start_date, end_date, train_df, feature_cols, best_lag)
    
    X_future = future_df[feature_cols].values
    X_future_scaled = scaler_X.transform(X_future)
    
    # Prophet prediction
    prophet_future = future_df[['Date', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']].copy()
    prophet_future.columns = ['ds', 'Airtemp', 'windspeed', 'RelHumidity', 'dewpointtemp']
    prophet_pred = prophet_model.predict(prophet_future)['yhat'].values
    
    # XGBoost prediction
    xgb_pred_scaled = xgb_model.predict(X_future_scaled)
    xgb_pred = scaler_y.inverse_transform(xgb_pred_scaled.reshape(-1, 1)).flatten()
    
    # N-BEATS prediction
    nbeats_model.eval()
    nbeats_preds = []
    with torch.no_grad():
        for i in range(len(X_future_scaled)):
            if i < lookback:
                X_input = np.concatenate([X_train_scaled[-(lookback-i):], X_future_scaled[:i]])
            else:
                X_input = X_future_scaled[i-lookback:i]
            
            X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
            pred = nbeats_model(X_tensor).cpu().numpy()[0, 0]
            nbeats_preds.append(pred)
    
    nbeats_pred = scaler_y.inverse_transform(np.array(nbeats_preds).reshape(-1, 1)).flatten()
    
    # DLinear prediction
    dlinear_model.eval()
    dlinear_preds = []
    with torch.no_grad():
        for i in range(len(X_future_scaled)):
            if i < lookback:
                X_input = np.concatenate([X_train_scaled[-(lookback-i):], X_future_scaled[:i]])
            else:
                X_input = X_future_scaled[i-lookback:i]
            
            X_tensor = torch.FloatTensor(X_input.flatten()).unsqueeze(0).to(device)
            pred = dlinear_model(X_tensor).cpu().numpy()[0, 0]
            dlinear_preds.append(pred)
    
    dlinear_pred = scaler_y.inverse_transform(np.array(dlinear_preds).reshape(-1, 1)).flatten()
    
    # Weighted ensemble prediction
    ensemble_pred = (
        ensemble_weights[0] * prophet_pred + 
        ensemble_weights[1] * xgb_pred + 
        ensemble_weights[2] * nbeats_pred + 
        ensemble_weights[3] * dlinear_pred
    )
    
    # Store monthly summary
    results_list.append({
        'Month': month_num,
        'Period': f"{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}",
        'Prophet_Mean': prophet_pred.mean(),
        'XGBoost_Mean': xgb_pred.mean(),
        'NBEATS_Mean': nbeats_pred.mean(),
        'DLinear_Mean': dlinear_pred.mean(),
        'Ensemble_Mean': ensemble_pred.mean(),
        'Prophet_Std': prophet_pred.std(),
        'XGBoost_Std': xgb_pred.std(),
        'NBEATS_Std': nbeats_pred.std(),
        'DLinear_Std': dlinear_pred.std(),
        'Ensemble_Std': ensemble_pred.std(),
    })
    
    # Store daily predictions
    for i, date in enumerate(future_df['Date']):
        detailed_forecasts.append({
            'Date': date,
            'Month_Ahead': month_num,
            'Prophet': prophet_pred[i],
            'XGBoost': xgb_pred[i],
            'NBEATS': nbeats_pred[i],
            'DLinear': dlinear_pred[i],
            'Ensemble': ensemble_pred[i]
        })

print("\nForecasting complete!")

results_df = pd.DataFrame(results_list)
detailed_df = pd.DataFrame(detailed_forecasts)

# ============================================================================
# SAVE RESULTS TO EXCEL
# ============================================================================

print("\nSaving results to Excel...")

output_path = os.path.join(desktop_path, "methane_forecast_results_UNIVERSAL.xlsx")

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Sheet 1: Forecast Summary
    results_df.to_excel(writer, sheet_name='Forecast_Summary', index=False)
    
    # Sheet 2: Validation Metrics
    validation_metrics.to_excel(writer, sheet_name='Validation_Metrics', index=False)
    
    # Sheet 3: Lag Correlation Analysis
    corr_df.to_excel(writer, sheet_name='Lag_Correlation', index=False)
    
    # Sheet 4: Feature Importance (XGBoost)
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    feature_importance.to_excel(writer, sheet_name='Feature_Importance', index=False)
    
    # Sheet 5: All Best Hyperparameters
    all_hyperparams = pd.DataFrame([
        {'Model': 'Prophet', 'Parameter': k, 'Value': v} 
        for k, v in study_prophet.best_params.items()
    ] + [
        {'Model': 'XGBoost', 'Parameter': k, 'Value': v} 
        for k, v in study_xgb.best_params.items()
    ] + [
        {'Model': 'N-BEATS', 'Parameter': k, 'Value': v} 
        for k, v in study_nbeats.best_params.items()
    ] + [
        {'Model': 'DLinear', 'Parameter': k, 'Value': v} 
        for k, v in study_dlinear.best_params.items()
    ])
    all_hyperparams.to_excel(writer, sheet_name='All_Hyperparameters', index=False)
    
    # Sheet 6: Daily Forecasts
    detailed_df.to_excel(writer, sheet_name='Daily_Forecasts', index=False)
    
    # Sheet 7: Ensemble Weights
    weights_df = pd.DataFrame({
        'Model': ['Prophet', 'XGBoost', 'N-BEATS', 'DLinear'],
        'Weight': ensemble_weights
    })
    weights_df.to_excel(writer, sheet_name='Ensemble_Weights', index=False)
    
    # Sheet 8: Model Configuration
    config_info = pd.DataFrame({
        'Parameter': [
            'Optimization Strategy',
            'Best Lag (days)',
            'Best Correlation',
            'Training Period Start',
            'Training Period End',
            'Training Samples',
            'Lookback Window (adaptive)',
            'Rolling Windows (adaptive)',
            'Scaler Type',
            'Prophet Trials',
            'XGBoost Trials',
            'N-BEATS Trials',
            'DLinear Trials',
            'Device',
            'Ensemble Method',
            'Dataset Days',
            'Adaptive Features'
        ],
        'Value': [
            'All models optimized with Optuna',
            int(best_lag),
            f"{best_corr:.4f}",
            str(train_df['Date'].min().date()),
            str(train_df['Date'].max().date()),
            len(train_df),
            lookback,
            str(windows),
            'RobustScaler',
            30,
            50,
            20,
            20,
            str(device),
            'Weighted by inverse RMSE',
            dataset_days,
            'Yes - adaptive to dataset size'
        ]
    })
    config_info.to_excel(writer, sheet_name='Model_Configuration', index=False)
    
    # Sheet 9: Optimization Scores
    optimization_scores = pd.DataFrame({
        'Model': ['Prophet', 'XGBoost', 'N-BEATS', 'DLinear'],
        'Best_Validation_R2': [
            study_prophet.best_value,
            study_xgb.best_value,
            study_nbeats.best_value,
            study_dlinear.best_value
        ]
    })
    optimization_scores.to_excel(writer, sheet_name='Optimization_Scores', index=False)

print(f"\n✓ Results saved to: {output_path}")

# ============================================================================
# SUMMARY AND RECOMMENDATIONS
# ============================================================================

print("\n" + "=" * 80)
print("UNIVERSAL FORECASTING COMPLETE - SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset Characteristics:")
print(f"   • Dataset days: {dataset_days}")
print(f"   • Optimal lag: {int(best_lag)} days (correlation: {best_corr:.4f})")
print(f"   • Lookback window: {lookback} days (adaptive)")
print(f"   • Rolling windows: {windows} (adaptive)")
print(f"   • Total features: {len(feature_cols)}")

print(f"\n🎯 Optimization Results:")
print(f"   • Prophet R²: {study_prophet.best_value:.4f} (30 trials)")
print(f"   • XGBoost R²: {study_xgb.best_value:.4f} (50 trials)")
print(f"   • N-BEATS R²: {study_nbeats.best_value:.4f} (20 trials)")
print(f"   • DLinear R²: {study_dlinear.best_value:.4f} (20 trials)")

print(f"\n📈 Validation Performance:")
for period in validation_metrics['Period'].unique():
    period_data = validation_metrics[validation_metrics['Period'] == period]
    print(f"\n  {period}:")
    for _, row in period_data.iterrows():
        print(f"    {row['Model']:12s}: R²={row['R2']:.4f}, RMSE={row['RMSE']:.4f}")

# Calculate average performance
avg_performance = validation_metrics.groupby('Model')[['R2', 'RMSE']].mean()
print(f"\n  Average Across All Periods:")
for model in avg_performance.index:
    print(f"    {model:12s}: R²={avg_performance.loc[model, 'R2']:.4f}, RMSE={avg_performance.loc[model, 'RMSE']:.4f}")

best_avg_model = avg_performance['R2'].idxmax()
best_avg_r2 = avg_performance['R2'].max()
print(f"\n  🏆 Most Consistent Model: {best_avg_model} (Avg R²={best_avg_r2:.4f})")

print(f"\n⚖️ Ensemble Strategy:")
print(f"   • Prophet: {ensemble_weights[0]:.3f}")
print(f"   • XGBoost: {ensemble_weights[1]:.3f}")
print(f"   • N-BEATS: {ensemble_weights[2]:.3f}")
print(f"   • DLinear: {ensemble_weights[3]:.3f}")

print(f"\n📅 Forecast Period: January 2025 - June 2025")
print(f"   • Daily predictions: {len(detailed_df)} days")
print(f"   • Recommended: Use Ensemble predictions")

print(f"\n✅ Why This Works Across ALL Datasets:")
print(f"   ✓ ALL models optimized equally (no bias)")
print(f"   ✓ Adaptive lag detection per dataset")
print(f"   ✓ Adaptive rolling windows based on data size")
print(f"   ✓ Adaptive lookback window")
print(f"   ✓ Multiple validation periods test consistency")
print(f"   ✓ Ensemble weights calculated per dataset")
print(f"   ✓ RobustScaler handles different scales")
print(f"   ✓ Early stopping prevents overfitting")
print(f"   ✓ Cross-validation ensures stability")
print(f"   ✓ No hardcoded dataset-specific parameters")

print(f"\n📁 Output File: methane_forecast_results_UNIVERSAL.xlsx")
print(f"   Contains 9 sheets with comprehensive analysis")

print("\n" + "=" * 80)
print("SUCCESS! This version adapts to ANY dataset automatically.")
print("=" * 80)

METHANE TIME SERIES FORECASTING SYSTEM
TRULY ROBUST VERSION - WORKS ACROSS ALL DATASETS

[1/10] Loading data from: C:\Users\persian\Desktop\All.xlsx
Dataset shape: (741, 6)
Columns: ['Date', 'windspeed', 'RelHumidity', 'dewpointtemp', 'Airtemp', 'methane']

First few rows:
        Date  windspeed  RelHumidity  dewpointtemp  Airtemp   methane
0 2019-01-04         14    59.533913             0       11  1884.257
1 2019-01-22         16    43.171667            -4       10  1900.618
2 2019-01-23          8    49.442083            -4        9  1886.202
3 2019-01-26         16    40.711667            -1       14  1885.500
4 2019-02-01         24    33.413750            -3       13  1850.711

Data statistics:
                                Date   windspeed  RelHumidity  dewpointtemp  \
count                            741  741.000000   741.000000    741.000000   
mean   2022-04-24 23:13:21.619433216   12.399460    27.696676      4.361673   
min              2019-01-04 00:00:00    4.000000   

14:14:58 - cmdstanpy - INFO - Chain [1] start processing
14:14:59 - cmdstanpy - INFO - Chain [1] done processing
14:14:59 - cmdstanpy - INFO - Chain [1] start processing
14:14:59 - cmdstanpy - INFO - Chain [1] done processing
14:14:59 - cmdstanpy - INFO - Chain [1] start processing
14:14:59 - cmdstanpy - INFO - Chain [1] done processing
14:15:00 - cmdstanpy - INFO - Chain [1] start processing
14:15:00 - cmdstanpy - INFO - Chain [1] done processing
14:15:00 - cmdstanpy - INFO - Chain [1] start processing
14:15:00 - cmdstanpy - INFO - Chain [1] done processing
14:15:01 - cmdstanpy - INFO - Chain [1] start processing
14:15:01 - cmdstanpy - INFO - Chain [1] done processing
14:15:01 - cmdstanpy - INFO - Chain [1] start processing
14:15:01 - cmdstanpy - INFO - Chain [1] done processing
14:15:01 - cmdstanpy - INFO - Chain [1] start processing
14:15:01 - cmdstanpy - INFO - Chain [1] done processing
14:15:02 - cmdstanpy - INFO - Chain [1] start processing
14:15:02 - cmdstanpy - INFO - Chain [1]

Best Prophet score: 0.4407
Best parameters: {'changepoint_prior_scale': 0.002713347673180981, 'seasonality_prior_scale': 0.07661726271819858, 'seasonality_mode': 'additive', 'changepoint_range': 0.8395289308025002}


14:15:10 - cmdstanpy - INFO - Chain [1] start processing
14:15:10 - cmdstanpy - INFO - Chain [1] done processing


Prophet model trained successfully!

[6/10] Optimizing XGBoost...
Best XGBoost score: 0.4880
Best parameters: {'max_depth': 8, 'learning_rate': 0.02176394646633798, 'n_estimators': 378, 'min_child_weight': 1, 'subsample': 0.6576540053559812, 'colsample_bytree': 0.8783890280805334, 'gamma': 0.8284429470405393, 'reg_alpha': 0.020929397651447468, 'reg_lambda': 0.6123292997710453}
XGBoost model trained successfully!

[7/10] Optimizing N-BEATS...
Best N-BEATS score: 0.2476
Best parameters: {'n_blocks': 2, 'n_layers': 3, 'n_neurons': 128, 'dropout': 0.2883520916464449, 'lr': 0.00017412125060705264}
Using device: cpu
Epoch 10/100, Loss: 0.277016
Epoch 20/100, Loss: 0.166661
Epoch 30/100, Loss: 0.091972
Epoch 40/100, Loss: 0.067772
Epoch 50/100, Loss: 0.048499
Epoch 60/100, Loss: 0.035956
Epoch 70/100, Loss: 0.033724
Epoch 80/100, Loss: 0.029350
Epoch 90/100, Loss: 0.024215
Epoch 100/100, Loss: 0.020000
N-BEATS model trained successfully!

[8/10] Optimizing DLinear...
Best DLinear score: 0.306